# 06 — XGBoost Regression: Predicting Chega Vote Share Change
Regression model to predict the magnitude of Chega's vote share change (not just direction) using XGBoost with Leave-One-Out cross-validation.


In [ ]:
#%pip install xgboost

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('tabela_principal_autarquicas_2025_v0.csv', sep=';')

In [ ]:
# 1️⃣ Preparar dados
# Calcular crescimento percentual do Chega
df["crescimento_chega_pct"] = df["CH_2022"] - df["CH_2021"]

# Variável alvo
y = df["crescimento_chega_pct"]

# Variáveis preditoras: todas exceto Concelho e colunas de votos do Chega
X = df.drop(columns=["Concelho", "CH_2021", "CH_2022", "crescimento_chega_pct"])

# 🔹 Limpar nomes das colunas para XGBoost
X_cleaned = X.copy()
X_cleaned.columns = (
    X_cleaned.columns
    .str.replace(r"[\[\]\<\>/]", "_", regex=True)  # substituir [], <, >, /
    .str.replace(" ", "_")                          # substituir espaços por _
    .str.replace("-", "_")                          # substituir - por _
)
X = X_cleaned

# 2️⃣ Inicializar Leave-One-Out
loo = LeaveOneOut()
y_true, y_pred = [], []

# 3️⃣ Loop LOO
for train_idx, test_idx in loo.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Modelo XGBoost Regressor
    model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective="reg:squarederror"
    )

    # Treinar modelo
    model.fit(X_train, y_train)

    # Prever crescimento percentual do concelho de teste
    y_hat = model.predict(X_test)
    y_true.append(y_test.values[0])
    y_pred.append(y_hat[0])

# 4️⃣ Avaliar performance
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)

print(f"LOO RMSE: {rmse:.3f}")
print(f"LOO R²: {r2:.3f}")

# 5️⃣ Criar tabela final com previsão para cada concelho
df_result = df[["Concelho"]].copy()
df_result["crescimento_previsto_2025"] = y_pred
df_result = df_result.sort_values(by="crescimento_previsto_2025", ascending=False)

print(df_result.head(10))  # top 10 concelhos com maior crescimento previsto
